In [145]:
import pandas as pd
import numpy as np

# Реальные средние температуры (примерные данные) для городов по сезонам
seasonal_temperatures = {
    "New York": {"winter": 0, "spring": 10, "summer": 25, "autumn": 15},
    "London": {"winter": 5, "spring": 11, "summer": 18, "autumn": 12},
    "Paris": {"winter": 4, "spring": 12, "summer": 20, "autumn": 13},
    "Tokyo": {"winter": 6, "spring": 15, "summer": 27, "autumn": 18},
    "Moscow": {"winter": -10, "spring": 5, "summer": 18, "autumn": 8},
    "Sydney": {"winter": 12, "spring": 18, "summer": 25, "autumn": 20},
    "Berlin": {"winter": 0, "spring": 10, "summer": 20, "autumn": 11},
    "Beijing": {"winter": -2, "spring": 13, "summer": 27, "autumn": 16},
    "Rio de Janeiro": {"winter": 20, "spring": 25, "summer": 30, "autumn": 25},
    "Dubai": {"winter": 20, "spring": 30, "summer": 40, "autumn": 30},
    "Los Angeles": {"winter": 15, "spring": 18, "summer": 25, "autumn": 20},
    "Singapore": {"winter": 27, "spring": 28, "summer": 28, "autumn": 27},
    "Mumbai": {"winter": 25, "spring": 30, "summer": 35, "autumn": 30},
    "Cairo": {"winter": 15, "spring": 25, "summer": 35, "autumn": 25},
    "Mexico City": {"winter": 12, "spring": 18, "summer": 20, "autumn": 15},
}

# Сопоставление месяцев с сезонами
month_to_season = {12: "winter", 1: "winter", 2: "winter",
                   3: "spring", 4: "spring", 5: "spring",
                   6: "summer", 7: "summer", 8: "summer",
                   9: "autumn", 10: "autumn", 11: "autumn"}

# Генерация данных о температуре
def generate_realistic_temperature_data(cities, num_years=10):
    dates = pd.date_range(start="2010-01-01", periods=365 * num_years, freq="D")
    data = []

    for city in cities:
        for date in dates:
            season = month_to_season[date.month]
            mean_temp = seasonal_temperatures[city][season]
            # Добавляем случайное отклонение
            temperature = np.random.normal(loc=mean_temp, scale=5)
            data.append({"city": city, "timestamp": date, "temperature": temperature})

    df = pd.DataFrame(data)
    df['season'] = df['timestamp'].dt.month.map(lambda x: month_to_season[x])
    return df

# Генерация данных
data = generate_realistic_temperature_data(list(seasonal_temperatures.keys()))
data.to_csv('temperature_data.csv', index=False)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Провести **анализ временных рядов**

Вычисление скользящего среднего и стандартного отклонения для сглаживания температурных колебаний:

In [140]:
data = data.sort_values(by=['city', 'timestamp'], ascending=[True, True])
data['temperature_rolling_mean'] = data.groupby(['city', 'season'])['temperature'].transform(lambda x: x.rolling(window=30, min_periods=1).mean())

data.head(5)

,city,timestamp,temperature,season,temperature_rolling_mean
25550,Beijing,2010-01-01,2.032879,winter,2.032879
25551,Beijing,2010-01-02,-13.148572,winter,-5.557846
25552,Beijing,2010-01-03,-0.307703,winter,-3.807799
25553,Beijing,2010-01-04,-3.098122,winter,-3.630379
25554,Beijing,2010-01-05,3.462238,winter,-2.211856


Определение аномалий на основе отклонений температуры от $ \text{скользящее среднее} \pm 2\sigma $

Создадим колонку is_anomaly_temperature, которая будет принимать значение True, если температура записи является аномальной. False - в противном случае

In [141]:
data['temperature_std'] = data.groupby(['city', 'season'])['temperature'].transform(lambda x: x.std())

data['is_anomaly_temperature'] = np.where((data['temperature'] > data['temperature_rolling_mean'] + 2 * data['temperature_std'] )|(data['temperature'] < data['temperature_rolling_mean'] - 2 * data['temperature_std']), True, False)
data[data['is_anomaly_temperature'] == True]

,city,timestamp,temperature,season,temperature_rolling_mean,temperature_std,is_anomaly_temperature
25573,Beijing,2010-01-24,10.455761,winter,-3.044790,4.970850,True
25584,Beijing,2010-02-04,8.761666,winter,-1.924474,4.970850,True
25588,Beijing,2010-02-08,-12.286031,winter,-1.494293,4.970850,True
25592,Beijing,2010-02-12,-11.735796,winter,-1.444058,4.970850,True
25601,Beijing,2010-02-21,-10.509194,winter,-0.199956,4.970850,True
...,...,...,...,...,...,...,...
14444,Tokyo,2019-07-27,13.766733,summer,26.600820,5.022677,True
14498,Tokyo,2019-09-19,7.824033,autumn,19.534478,4.913350,True
14501,Tokyo,2019-09-22,8.920217,autumn,19.447653,4.913350,True
14538,Tokyo,2019-10-29,6.021545,autumn,17.873893,4.913350,True


Построение долгосрочных трендов изменения температуры:

Используем линейную регрессию для прогнозирования дальнейшей температуры
Закодируем сезон и город с помощью OneHotEncoder, timestamp переведем в число (количество дней от начала нашей эры)

Полученную модель можно использовать для предсказания дальнейшей температуры

In [142]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import datetime as dt

def prepare_x_data(data_to_prepare, encoder):
  season_city_encoded = encoder.transform(data_to_prepare[['season', 'city']])
  season_city_encoded_df = pd.DataFrame(season_city_encoded, columns=encoder.get_feature_names_out(['season', 'city']))

  X = pd.concat([data_to_prepare['timestamp'].map(dt.datetime.toordinal), season_city_encoded_df], axis=1)
  return X

encoder = OneHotEncoder(sparse_output=False, drop='first')
encoder.fit(data[['season', 'city']])
X = prepare_x_data(data, encoder)
y = data['temperature']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print(f'''Metrics:
R2 = {r2_score(y_test, y_pred)}
''')

Metrics:
R2 = 0.7303415304869828



Осуществить **мониторинг текущей температуры**

In [143]:
import requests

api_key = ""

def get_current_temperature(city):
    try:
        params = {"q": city,
              "appid": api_key,
              "units": "metric"}
        response = requests.get(
            "https://api.openweathermap.org/data/2.5/weather",
            params=params)

        data = response.json()
        temperature = data["main"]["temp"]
        time = data["sys"]["sunrise"]
        month = pd.to_datetime(time, unit='s').month
        return temperature, month
    except requests.exceptions.RequestException as e:
        raise f"Exception while sending request: {e}"

def is_temperature_anomal(city, temperature, month, data):
    data_filtered = data[(data["city"] == city) & (data["timestamp"].dt.month == month)]
    min = data_filtered.iloc[0]['temperature_rolling_mean'] - 2 * data_filtered.iloc[0]['temperature_std']
    max = data_filtered.iloc[0]['temperature_rolling_mean'] + 2 * data_filtered.iloc[0]['temperature_std']
    return (temperature > max) or (temperature < min)

def get_mean_temperature(city, temperature, month, data):
    data_filtered = data[(data["city"] == city) & (data["timestamp"].dt.month == month)]
    return data_filtered.iloc[0]['temperature_rolling_mean']

def define_anomal_temperature(city):
  temperature, month = get_current_temperature(city)
  print(f"Current temperature in {city}: {temperature}°C")

  is_anomaly = is_temperature_anomal(city, temperature, month, data)
  if is_anomaly:
    mean_temperature = get_mean_temperature(city, temperature, month, data)
    print(f"Current temperature in {city} is anomal, mean temperature = {mean_temperature}")
  else:
    print(f"Current temperature in {city} is normal")
  print()

define_anomal_temperature('Moscow')
define_anomal_temperature('London')
define_anomal_temperature('Beijing')
define_anomal_temperature('Tokyo')
define_anomal_temperature('New York')

Current temperature in Moscow: -0.52°C
Current temperature in Moscow is normal

Current temperature in London: 2.54°C
Current temperature in London is normal

Current temperature in Beijing: -7.06°C
Current temperature in Beijing is normal

Current temperature in Tokyo: 4.41°C
Current temperature in Tokyo is normal

Current temperature in New York: 5.03°C
Current temperature in New York is normal



Погода в Лондоне, Пекине, Токио и Нью-Йорке в пределах нормы. Погода в Москве аномальна (оно в целом видно и по наблюдениям)